# Binance daily SVM grid search: step-by-step walkthrough

This notebook explains and executes the approved BTCUSDT development grid search one stage at a time. It reuses the repository implementation in `src/tcc_crypto/daily_svm_grid_search.py`; the notebook does not create an alternative modeling pipeline.

The walkthrough covers configuration, development-only data, target balance, asset eligibility, temporal folds, fold-local feature selection and scaling, conditional SVC candidates, model ranking, selected configurations, and persisted evidence.

This walkthrough uses the shared daily stages and centralized artifact paths. Install the project with `python -m pip install -e ".[dev]"` before opening the notebook. Outputs were cleared after refactoring; execute cells in order to generate current evidence. `OVERWRITE` defaults to `False`. No final-test evaluator is executed here.


## Methodological boundaries

- Feature dates in development run from 2019-07-01 through 2024-12-30.
- The 2024-12-31 candle is used only to construct the target for the 2024-12-30 feature row.
- Final-test feature rows beginning on 2025-01-01 are not retained, calculated, or used.
- Inputs at `t` map to the next day's same-candle return: `features(t) -> D0(t + 1)`.
- Searches are independent for each symbol, target, and 60-, 75-, or 90-day training window.
- Every inner split has a 60-observation validation block and a one-observation gap.
- Feature selection and `StandardScaler` are fitted only on each inner training fold.
- Mean validation F1 is the primary selection metric. The final test is outside this notebook.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


from tcc_crypto.paths import repository_root
from tcc_crypto.workflows.context import load_context
from tcc_crypto.workflows import daily_data
from tcc_crypto.workflows.svm import persist_results
from tcc_crypto.features.catalog import ACTIVE_FEATURES, feature_definitions
from tcc_crypto.features.warmup import remove_leading_feature_warmup
from tcc_crypto.data.dataset import sample_features as select_sample_features
from tcc_crypto.modeling.preprocessing import prepare_window

REPOSITORY_ROOT = repository_root()
from tcc_crypto.modeling.config import expand_parameter_grid
from tcc_crypto.modeling.splits import temporal_splits
from tcc_crypto.modeling.eligibility import assess_symbol_eligibility
from tcc_crypto.modeling.grid_search import run_temporal_grid_search, evaluate_candidate

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)
print(f'Repository root: {REPOSITORY_ROOT}')

## 1. Load and validate the approved configuration

The JSON configuration is the source of executable experiment parameters. The loader rejects changes that conflict with the approved methodology, such as random shuffling, different dates, or a grid with the wrong number of candidates.

In [ ]:
symbol = 'BTCUSDT'
context = load_context('development', symbol=symbol, root=REPOSITORY_ROOT)
grid_config = context.model
feature_config = context.features
OVERWRITE = False  # Set True explicitly to replace generated artifacts.
display(pd.Series(context.paths.inventory(context.kind), name='resolved_path').to_frame())
display(pd.DataFrame(feature_definitions()))

configuration_overview = pd.Series(
    {
        'experiment_id': grid_config.experiment_id,
        'development_start_utc': grid_config.development_start_utc,
        'development_end_utc': grid_config.development_end_utc,
        'targets': grid_config.targets,
        'training_window_sizes': grid_config.training_window_sizes,
        'inner_splits': grid_config.n_splits,
        'validation_block_size': grid_config.test_size,
        'gap_observations': grid_config.gap,
        'minimum_complete_rows': grid_config.minimum_complete_rows,
        'candidate_count_per_search': grid_config.candidate_count,
        'primary_metric': grid_config.primary_metric,
        'explicitly_excluded_symbols': grid_config.excluded_symbols,
    },
    name='approved_value',
)
display(configuration_overview.to_frame())

## 2. Expand the conditional hyperparameter grid

The grid is conditional: parameters appear only for kernels that use them. This produces 6 linear, 18 RBF, and 162 polynomial candidates, for 186 candidates per symbol-target-window search.

In [ ]:
parameter_candidates = expand_parameter_grid(grid_config.parameter_grid)
parameter_table = pd.DataFrame(parameter_candidates)
parameter_table.insert(0, 'candidate_order', range(len(parameter_table)))

branch_counts = (
    parameter_table.groupby('kernel', sort=False)
    .size()
    .rename('candidate_count')
    .to_frame()
)
display(branch_counts)
display(parameter_table.head(10))
print(f'Total compatible candidates: {len(parameter_table)}')

## 3. Build the BTCUSDT development dataset

This calls the same development-data builder used by the command-line runner. It applies the approved internal-gap treatment in derived data, calculates the 22 features, removes only leading indicator warm-up rows, and constructs the three next-day targets.

In [ ]:
# Each step can also be run separately from the command line.
daily_data.align(context, overwrite=OVERWRITE)
daily_data.features(context, overwrite=OVERWRITE)
daily_data.targets(context, overwrite=OVERWRITE)
daily_data.dataset(context, overwrite=OVERWRITE)
development_dataset = daily_data.read_stage(context, 'dataset')

dataset_overview = pd.Series(
    {
        'symbol': symbol,
        'raw_source': daily_data.raw_source(context),
        'development_rows': len(development_dataset),
        'feature_count': len(ACTIVE_FEATURES),
        'first_feature_date_utc': development_dataset.index.min(),
        'last_feature_date_utc': development_dataset.index.max(),
        'latest_target_date_utc': development_dataset['target_date_utc'].max(),
        'feature_imputed_rows': int(development_dataset['feature_is_imputed'].sum()),
        'target_imputed_rows': int(development_dataset['target_is_imputed'].fillna(False).sum()),
        'missing_active_feature_cells': int(development_dataset.loc[:, ACTIVE_FEATURES].isna().sum().sum()),
    },
    name='development_dataset',
)
display(dataset_overview.to_frame())
display(development_dataset.head(5))

## 4. Inspect target balance

Higher return thresholds create fewer positive observations. Accuracy alone can therefore look acceptable even when a classifier rarely detects the positive class, which is why F1, precision, recall, ROC-AUC, and average precision are retained.

In [ ]:
target_balance_rows = []
for target in grid_config.targets:
    labels = development_dataset[target].dropna().astype('int64')
    target_balance_rows.append(
        {
            'target': target,
            'negative': int((labels == 0).sum()),
            'positive': int((labels == 1).sum()),
            'positive_rate': float((labels == 1).mean()),
        }
    )
target_balance = pd.DataFrame(target_balance_rows).set_index('target')
display(target_balance)

ax = target_balance[['negative', 'positive']].plot.bar(
    figsize=(8, 4), color=['#4472C4', '#ED7D31']
)
ax.set_title('BTCUSDT development target-class balance')
ax.set_xlabel('Binary target')
ax.set_ylabel('Daily observations')
ax.legend(title='Class')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Evaluate search eligibility

Eligibility is checked separately for every target and training window. A search needs at least 271 complete labeled rows and both classes in every inner training fold. Exclusions and failures are recorded instead of being silently discarded.

In [ ]:
eligibility = assess_symbol_eligibility(
    development_dataset,
    symbol=symbol,
    config=grid_config,
)
display(
    eligibility.loc[
        :,
        [
            'symbol',
            'target',
            'window_size',
            'status',
            'reason',
            'complete_labeled_rows',
            'negative_count',
            'positive_count',
        ],
    ]
)

## 6. Visualize the temporal split boundaries

The example below uses the 90-day window. Each validation block contains 60 later observations. The one row between training and validation is excluded as the gap. No row is shuffled.

In [ ]:
example_target = 'target_2pct'
example_window_size = 90
example_complete = development_dataset.loc[:, [*ACTIVE_FEATURES, example_target]].dropna()
example_splits = temporal_splits(len(example_complete), grid_config, example_window_size)

split_rows = []
for fold, (training_positions, validation_positions) in enumerate(example_splits):
    gap_start = training_positions[-1] + 1
    split_rows.append(
        {
            'fold': fold,
            'training_start_utc': example_complete.index[training_positions[0]],
            'training_end_utc': example_complete.index[training_positions[-1]],
            'training_rows': len(training_positions),
            'gap_date_utc': example_complete.index[gap_start],
            'validation_start_utc': example_complete.index[validation_positions[0]],
            'validation_end_utc': example_complete.index[validation_positions[-1]],
            'validation_rows': len(validation_positions),
        }
    )
split_boundaries = pd.DataFrame(split_rows)
display(split_boundaries)

## 7. Inspect preprocessing inside one fold

The selector is fitted only on the 90 training rows. The validation rows are transformed with that fitted selector. Then `StandardScaler` is fitted only on the selected training matrix and applied unchanged to validation.

In [ ]:
training_positions, validation_positions = example_splits[0]
feature_matrix = example_complete.loc[:, ACTIVE_FEATURES].astype('float64')
labels = example_complete[example_target].astype('int64')
training_features = feature_matrix.iloc[training_positions]
validation_features = feature_matrix.iloc[validation_positions]
training_labels = labels.iloc[training_positions]
validation_labels = labels.iloc[validation_positions]

prepared = prepare_window(training_features, validation_features, feature_config.feature_selection)
selector = prepared.selector
selected_training = selector.transform(training_features)
scaler = prepared.scaler
scaled_training = prepared.training
scaled_validation = prepared.evaluation

preprocessing_overview = pd.Series(
    {
        'input_feature_count': training_features.shape[1],
        'selected_feature_count': len(selector.selected_features_),
        'selected_features': ', '.join(selector.selected_features_),
        'training_rows_used_to_fit': len(training_features),
        'validation_rows_only_transformed': len(validation_features),
        'training_negative': int((training_labels == 0).sum()),
        'training_positive': int((training_labels == 1).sum()),
    },
    name='fold_0_preprocessing',
)
display(preprocessing_overview.to_frame())

scaled_training_frame = pd.DataFrame(
    scaled_training, columns=selected_training.columns, index=selected_training.index
)
display(
    pd.DataFrame(
        {
            'training_mean_after_scaling': scaled_training_frame.mean(),
            'training_std_after_scaling': scaled_training_frame.std(ddof=0),
        }
    )
)

## 8. Compare one candidate from each kernel branch

This educational cell fits three candidates on the same preprocessed fold. The official search repeats this for every compatible candidate and every temporal fold. ROC-AUC and average precision use the continuous `decision_function` score, not the binary prediction.

In [ ]:
example_candidate_orders = [0, 6, 24]
example_candidate_rows = []
for candidate_order in example_candidate_orders:
    parameters = parameter_candidates[candidate_order]
    metrics = evaluate_candidate(parameters, training_labels, validation_labels,
                                 scaled_training, scaled_validation)
    example_candidate_rows.append(
        {
            'candidate_order': candidate_order,
            **parameters,
            **metrics,
        }
    )
example_candidate_comparison = pd.DataFrame(example_candidate_rows)
display(example_candidate_comparison)

## 9. Run the complete approved BTCUSDT grid search

This is the complete development search: 186 candidates × 3 folds × 3 targets × 3 training windows = 5,022 candidate-fold fits. It normally completes in under one minute on the development machine. It still does not execute the final test.

In [ ]:
grid_result = run_temporal_grid_search(
    development_dataset,
    symbol=symbol,
    config=grid_config,
    selection_config=feature_config.feature_selection,
)

run_overview = pd.Series(
    {
        'eligible_searches': int((grid_result.eligibility['status'] == 'eligible').sum()),
        'candidate_fold_fits': len(grid_result.fold_results),
        'successful_fits': int((grid_result.fold_results['status'] == 'ok').sum()),
        'fit_errors': int((grid_result.fold_results['status'] != 'ok').sum()),
        'aggregate_candidate_rows': len(grid_result.candidate_results),
        'selected_configurations': len(grid_result.selected_configurations),
        'latest_validation_date_utc': grid_result.fold_results['validation_end_utc'].max(),
    },
    name='complete_grid_search',
)
display(run_overview.to_frame())

## 10. Inspect fold-level evidence

Each row below is one candidate evaluated on one fold. It records temporal boundaries, class counts, selected features, metrics, diagnostics, and fit duration.

In [ ]:
fold_columns = [
    'target',
    'window_size',
    'candidate_order',
    'fold',
    'kernel',
    'C',
    'training_start_utc',
    'training_end_utc',
    'validation_start_utc',
    'validation_end_utc',
    'selected_feature_count',
    'accuracy',
    'precision',
    'recall',
    'f1',
    'roc_auc',
    'average_precision',
    'status',
]
display(grid_result.fold_results.loc[:, fold_columns].head(12))

## 11. Inspect candidate aggregation and deterministic ranking

Fold metrics are averaged for each candidate. The primary ranking is mean F1. Candidates within the configured F1 tolerance are ordered by precision, F1 stability, ROC-AUC, kernel simplicity, lower C, and finally configured grid order.

In [ ]:
ranking_columns = [
    'target',
    'window_size',
    'selection_rank',
    'candidate_order',
    'kernel',
    'C',
    'gamma',
    'degree',
    'coef0',
    'class_weight',
    'mean_f1',
    'mean_precision',
    'f1_standard_deviation',
    'mean_roc_auc',
]
top_candidates = (
    grid_result.candidate_results
    .sort_values(['target', 'window_size', 'selection_rank'])
    .groupby(['target', 'window_size'], sort=True)
    .head(3)
)
display(top_candidates.loc[:, ranking_columns])

## 12. View the selected configuration for every target and window

These are development-period selections. They are not final out-of-sample performance estimates. The hyperparameters must remain frozen before the later final walk-forward evaluation.

In [ ]:
selected_columns = [
    'target',
    'window_size',
    'kernel',
    'C',
    'gamma',
    'degree',
    'coef0',
    'class_weight',
    'mean_accuracy',
    'mean_precision',
    'mean_recall',
    'mean_f1',
    'mean_roc_auc',
    'mean_average_precision',
]
selected = grid_result.selected_configurations.loc[:, selected_columns].copy()
display(selected)

plot_data = selected.pivot(index='window_size', columns='target', values='mean_f1')
ax = plot_data.plot(marker='o', figsize=(8, 4))
ax.set_title('Selected candidate mean validation F1 — BTCUSDT development')
ax.set_xlabel('Training-window size (daily observations)')
ax.set_ylabel('Mean validation F1')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
ax.legend(title='Target')
plt.tight_layout()
plt.show()

## 13. Persist reproducibility evidence

The writer stores eligibility, fold results, candidate rankings, selected configurations, a configuration snapshot, and run metadata with checksums. These generated artifacts remain outside Git.

In [ ]:
persist_results(context, grid_result, overwrite=OVERWRITE)
artifact_table = pd.Series(context.paths.inventory(context.kind), name='path').to_frame()
display(artifact_table)


## 14. How to run other scopes

The notebook intentionally executes only the approved BTCUSDT operational validation. The command-line runner provides explicit expansion controls:

```bash
# BTCUSDT complete development grid search (default)
python scripts/run_daily_svm_grid_search.py

# Eligibility only for all 40 configured symbols
python scripts/run_daily_svm_grid_search.py --all-symbols --eligibility-only

# Complete development grid for all eligible symbol-target-window combinations
python scripts/run_daily_svm_grid_search.py --all-symbols
```

The final command is intentionally separate because it is substantially larger than the BTCUSDT validation. It still operates only on development data.

## Interpretation limits and next step

This notebook demonstrates model selection inside the development period. Its mean fold metrics compare hyperparameter candidates; they are not the definitive estimate of future performance. The reserved 2025-01-01 through 2026-08-31 feature period has not been evaluated here.

The next implementation stage is the frozen-parameter one-step walk-forward evaluator, including the constant-zero dummy, prior dummy, and default-SVC baselines. That stage must use the selected development parameters without revising the grid after final-test outcomes are observed.